**Load DA results**

In [1]:
import sys
import os
import numpy as np
import gurobipy as gp
from gurobipy import GRB
import json

# --- Path setup ---
hour = int(os.environ['HOUR'])

if 'data' in sys.modules:
    del sys.modules['data']

sys.path.insert(0, os.environ['PYTHONPATH'])

from data import load_distribution, load_profile, generators, generator_bid_prices
from data import Prices_for_loads

# --- Load Step 1 DA results from JSON ---
results_path = os.path.join(os.environ['PYTHONPATH'], 'Results', 'DA_results_step1.json')
with open(results_path, 'r') as f:
    DA_results_step1 = json.load(f)

lambda_DA      = DA_results_step1['lambda_DA_step1']
dispatch_step1 = DA_results_step1['dispatch_step1']

# --- Variable lists ---
VARIABLES      = list(generators.keys())
LOAD_VARIABLES = list(load_distribution.keys())
conv_names     = [g for g in VARIABLES if g.startswith('G')]
wind_names     = [g for g in VARIABLES if g.startswith('W')]

# --- Conventional generator costs, max caps, DA dispatch ---
gen_costs    = {g: generator_bid_prices[g][hour] for g in conv_names}
gen_max_caps = {g: generators[g]['Pmax_MW']      for g in conv_names}
g_DA         = {g: dispatch_step1.get(g, 0)      for g in conv_names}

# --- Wind DA scheduled output (from Step 1 dispatch) ---
Wind_DA = np.array([dispatch_step1.get(w, 0) for w in wind_names])

print(f"λ_DA = {lambda_DA:.2f} EUR/MWh")
print(f"DA total generation: {sum(g_DA.values()) + sum(Wind_DA):.2f} MW")

print(f"\nConventional dispatch:")
for g, v in g_DA.items():
    if v > 1e-3:
        print(f"  {g}: {v:.2f} MW")

print(f"\nWind dispatch:")
for wi, w in enumerate(wind_names):
    print(f"  {w}: {Wind_DA[wi]:.2f} MW")

λ_DA = 9.47 EUR/MWh
DA total generation: 2650.50 MW

Conventional dispatch:
  G6: 155.00 MW
  G7: 155.00 MW
  G8: 400.00 MW
  G9: 400.00 MW
  G10: 300.00 MW
  G11: 310.00 MW
  G12: 103.51 MW

Wind dispatch:
  W1: 131.63 MW
  W2: 143.28 MW
  W3: 142.54 MW
  W4: 124.68 MW
  W5: 144.88 MW
  W6: 139.97 MW


**Define BM deviations**

In [2]:
# --- Realisation deviations from DA schedule ---
# W1, W2, W3 produce 15% LESS than forecast (same direction as system short → bad)
# W4, W5, W6 produce 10% MORE than forecast (opposite direction → helps)
# G8 has a full outage (was scheduled, now produces 0)

Wind_actual = Wind_DA.copy()
for idx in [0, 1, 2]:
    Wind_actual[idx] = Wind_DA[idx] * 0.85   # 15% less
for idx in [3, 4, 5]:
    Wind_actual[idx] = Wind_DA[idx] * 1.10   # 10% more

wind_delta      = np.sum(Wind_actual) - np.sum(Wind_DA)
outage_delta    = 0 - g_DA['G8']
total_imbalance = wind_delta + outage_delta

print(f"Wind delta:       {wind_delta:+.2f} MW")
print(f"G8 outage delta:  {outage_delta:+.2f} MW")
print(f"Total imbalance:  {total_imbalance:+.2f} MW")
print(f"Direction needed: {'UPWARD regulation' if total_imbalance < 0 else 'DOWNWARD regulation'}")

print(f"\nWind actual vs DA scheduled:")
for wi, w in enumerate(wind_names):
    delta = Wind_actual[wi] - Wind_DA[wi]
    print(f"  {w}: DA={Wind_DA[wi]:.2f} MW  actual={Wind_actual[wi]:.2f} MW  delta={delta:+.2f} MW")

Wind delta:       -21.67 MW
G8 outage delta:  -400.00 MW
Total imbalance:  -421.67 MW
Direction needed: UPWARD regulation

Wind actual vs DA scheduled:
  W1: DA=131.63 MW  actual=111.89 MW  delta=-19.74 MW
  W2: DA=143.28 MW  actual=121.79 MW  delta=-21.49 MW
  W3: DA=142.54 MW  actual=121.16 MW  delta=-21.38 MW
  W4: DA=124.68 MW  actual=137.15 MW  delta=+12.47 MW
  W5: DA=144.88 MW  actual=159.37 MW  delta=+14.49 MW
  W6: DA=139.97 MW  actual=153.97 MW  delta=+14.00 MW


**BM regulation offers**

In [3]:
# --- Balancing offers ---
# Only conventional generators can participate in the balancing market
# G8 excluded due to outage
# Wind farms are NOT balancing providers — they are settled as deviators in the price scheme
# Per assignment: "A subset of conventional generators are potential balancing service providers"

BALANCING_SUBSET = {'G1', 'G2', 'G3', 'G4', 'G5', 'G6', 'G7',
                    'G9', 'G10', 'G11', 'G12'}

up_reg_offers   = {}
down_reg_offers = {}

for g in conv_names:
    if g not in BALANCING_SUBSET:
        continue

    c_i      = gen_costs[g]
    g_da_i   = g_DA[g]
    g_max_i  = gen_max_caps[g]

    # HEADROOM: max capacity - DA schedule = room to ramp up (eq. 25)
    headroom = g_max_i - g_da_i

    # FOOTROOM: DA schedule = room to ramp down (eq. 26)
    footroom = g_da_i

    # Upward reg offer: DA price + 10% of marginal cost (eq. 23)
    if headroom > 1e-3:
        up_reg_offers[g] = (lambda_DA + 0.10 * c_i, headroom)

    # Downward reg offer: DA price - 15% of marginal cost (eq. 23)
    if footroom > 1e-3:
        down_reg_offers[g] = (lambda_DA - 0.15 * c_i, footroom)

print("Upward regulation offers:")
for g, (p, mw) in up_reg_offers.items():
    print(f"  {g}: {p:.2f} EUR/MWh  (max {mw:.2f} MW)  "
          f"[ramp up from DA={g_DA[g]:.2f} MW to Pmax={gen_max_caps[g]:.2f} MW]")

print("\nDownward regulation offers:")
for g, (p, mw) in down_reg_offers.items():
    print(f"  {g}: {p:.2f} EUR/MWh  (max {mw:.2f} MW)  "
          f"[footroom below DA={g_DA[g]:.2f} MW]")

print(f"\nTotal upward available:   {sum(mw for p,mw in up_reg_offers.values()):.2f} MW")
print(f"Total downward available: {sum(mw for p,mw in down_reg_offers.values()):.2f} MW")
print(f"Imbalance to cover:       {-total_imbalance:.2f} MW")

Upward regulation offers:
  G1: 10.63 EUR/MWh  (max 152.00 MW)  [ramp up from DA=0.00 MW to Pmax=152.00 MW]
  G2: 10.63 EUR/MWh  (max 152.00 MW)  [ramp up from DA=0.00 MW to Pmax=152.00 MW]
  G3: 11.27 EUR/MWh  (max 350.00 MW)  [ramp up from DA=0.00 MW to Pmax=350.00 MW]
  G4: 11.29 EUR/MWh  (max 591.00 MW)  [ramp up from DA=0.00 MW to Pmax=591.00 MW]
  G5: 11.74 EUR/MWh  (max 60.00 MW)  [ramp up from DA=0.00 MW to Pmax=60.00 MW]
  G12: 10.42 EUR/MWh  (max 246.49 MW)  [ramp up from DA=103.51 MW to Pmax=350.00 MW]

Downward regulation offers:
  G6: 8.10 EUR/MWh  (max 155.00 MW)  [footroom below DA=155.00 MW]
  G7: 8.10 EUR/MWh  (max 155.00 MW)  [footroom below DA=155.00 MW]
  G9: 8.76 EUR/MWh  (max 400.00 MW)  [footroom below DA=400.00 MW]
  G10: 9.47 EUR/MWh  (max 300.00 MW)  [footroom below DA=300.00 MW]
  G11: 8.10 EUR/MWh  (max 310.00 MW)  [footroom below DA=310.00 MW]
  G12: 8.05 EUR/MWh  (max 103.51 MW)  [footroom below DA=103.51 MW]

Total upward available:   1551.49 MW
Total dow

**Optimize BM**

In [4]:
model_BM = gp.Model("BM_Hour")
model_BM.Params.OutputFlag     = 0
model_BM.Params.DualReductions = 0

# Upward regulation variables — bounded by each generator's headroom
r_up = {g: model_BM.addVar(lb=0, ub=mw, name=f'rup_{g}')
        for g, (p, mw) in up_reg_offers.items()}

# Downward regulation variables — bounded by each generator's footroom
r_dn = {g: model_BM.addVar(lb=0, ub=mw, name=f'rdn_{g}')
        for g, (p, mw) in down_reg_offers.items()}

# Load curtailment — last resort at 500 EUR/MWh
curtailment = model_BM.addVar(lb=0, name='curtailment')

# Balance constraint: net upward - net downward + curtailment = deficit to cover
model_BM.addLConstr(
    gp.quicksum(r_up[g] for g in r_up)
    - gp.quicksum(r_dn[g] for g in r_dn)
    + curtailment,
    GRB.EQUAL,
    -total_imbalance,
    name="BM_balance"
)

# Objective: minimise total balancing cost
obj = (gp.quicksum(up_reg_offers[g][0]   * r_up[g] for g in r_up)
     - gp.quicksum(down_reg_offers[g][0] * r_dn[g] for g in r_dn)
     + 500 * curtailment)

model_BM.setObjective(obj, GRB.MINIMIZE)
model_BM.optimize()

lambda_BM = model_BM.getConstrByName("BM_balance").Pi

print(f"\n{'='*50}")
print(f"BM clearing price: λ_BM = {lambda_BM:.2f} EUR/MWh")
print(f"Curtailment:       {curtailment.X:.2f} MW")

print(f"\nActivated upward regulation:")
for g in r_up:
    if r_up[g].X > 1e-3:
        if g_DA[g] < 1e-3:
            print(f"  {g}: +{r_up[g].X:.2f} MW at {up_reg_offers[g][0]:.2f} EUR/MWh  "
                  f"[not dispatched in DA, ramping from 0 to {r_up[g].X:.2f} MW]")
        else:
            print(f"  {g}: +{r_up[g].X:.2f} MW at {up_reg_offers[g][0]:.2f} EUR/MWh  "
                  f"[DA={g_DA[g]:.2f} MW → {g_DA[g] + r_up[g].X:.2f} MW]")

print(f"\nActivated downward regulation:")
for g in r_dn:
    if r_dn[g].X > 1e-3:
        print(f"  {g}: -{r_dn[g].X:.2f} MW at {down_reg_offers[g][0]:.2f} EUR/MWh  "
              f"[footroom below DA={g_DA[g]:.2f} MW]")

# Sanity check
total_up = sum(r_up[g].X for g in r_up)
total_dn = sum(r_dn[g].X for g in r_dn)
print(f"\nSanity check:")
print(f"  Total upward activated:   {total_up:.2f} MW")
print(f"  Total downward activated: {total_dn:.2f} MW")
print(f"  Curtailment:              {curtailment.X:.2f} MW")
print(f"  Net balance:              {total_up - total_dn + curtailment.X:.2f} MW  (should be {-total_imbalance:.2f} MW)")

Set parameter Username


Set parameter LicenseID to value 2658987


Academic license - for non-commercial use only - expires 2026-04-29



BM clearing price: λ_BM = 10.63 EUR/MWh
Curtailment:       0.00 MW

Activated upward regulation:
  G1: +152.00 MW at 10.63 EUR/MWh  [not dispatched in DA, ramping from 0 to 152.00 MW]
  G2: +23.18 MW at 10.63 EUR/MWh  [not dispatched in DA, ramping from 0 to 23.18 MW]
  G12: +246.49 MW at 10.42 EUR/MWh  [DA=103.51 MW → 350.00 MW]

Activated downward regulation:

Sanity check:
  Total upward activated:   421.67 MW
  Total downward activated: 0.00 MW
  Curtailment:              0.00 MW
  Net balance:              421.67 MW  (should be 421.67 MW)


**Profit under 1-price and 2-price settlement**

In [5]:
import matplotlib.pyplot as plt
import numpy as np

# ── ONE-PRICE vs TWO-PRICE SETTLEMENT ─────────────────────────────────────────
#
# Formulas from assignment (equations 28-31):
#
# (28) Conventional providers:
#      Profit = DA * PS_g + BP * p_gu - C_g * (PS_g + p_gu)
#
# (29) G8 outage causer:
#      Profit = DA * PS_g - BP * PS_g
#
# (30) Wind less than forecast:
#      BM settlement = - BP * |deviation|  (penalised in both schemes)
#
# (31) Wind more than forecast:
#      1-price: BM settlement = + BP * deviation  (rewarded at BP)
#      2-price: BM settlement = + DA * deviation  (no bonus, settled at DA)
#
# KEY DIFFERENCE: overproducing wind gets λ_BM under 1-price, λ_DA under 2-price
# ──────────────────────────────────────────────────────────────────────────────

def clean(x):
    return 0.0 if abs(x) < 1e-6 else x

print(f"λ_DA  = {lambda_DA:.2f} EUR/MWh")
print(f"λ_BM  = {lambda_BM:.2f} EUR/MWh")
print(f"System imbalance: {total_imbalance:+.2f} MW ({'SHORT - upward needed' if total_imbalance < 0 else 'LONG - downward needed'})")

# --- Deviations from DA schedule ---
dev_conv = {g: (0 - g_DA[g] if g == 'G8' else 0.0) for g in conv_names}
dev_wind = {w: Wind_actual[wi] - Wind_DA[wi] for wi, w in enumerate(wind_names)}

print(f"\nDeviations from DA schedule:")
for g, d in dev_conv.items():
    if abs(d) > 1e-3:
        print(f"  {g}: {d:+.2f} MW")
for w, d in dev_wind.items():
    print(f"  {w}: {d:+.2f} MW")

# --- DA profit (from JSON) ---
da_profits = {**DA_results_step1['generator_profits_step1']}

# ── ONE-PRICE SETTLEMENT ──────────────────────────────────────────────────────
print(f"\n{'='*60}")
print(f"ONE-PRICE SETTLEMENT (λ_BM = {lambda_BM:.2f} EUR/MWh)")
print(f"{'='*60}")

total_profit_1price = {}

# Conventional generators
for g in conv_names:
    ps_g = g_DA[g]
    p_gu = r_up[g].X if g in r_up else 0.0
    c_g  = gen_costs[g]

    if g == 'G8':
        # eq (29): earned DA for schedule, penalised at BP for full outage
        profit  = clean(lambda_DA * ps_g - lambda_BM * ps_g)
        formula = f"DA*PS - BP*PS = {lambda_DA:.2f}*{ps_g:.2f} - {lambda_BM:.2f}*{ps_g:.2f}"
    else:
        # eq (28): DA revenue + BP activation - production cost
        profit  = clean(lambda_DA * ps_g + lambda_BM * p_gu - c_g * (ps_g + p_gu))
        formula = (f"DA*PS + BP*p_gu - C_g*(PS+p_gu) = "
                  f"{lambda_DA:.2f}*{ps_g:.2f} + {lambda_BM:.2f}*{p_gu:.2f} "
                  f"- {c_g:.2f}*({ps_g:.2f}+{p_gu:.2f})")

    total_profit_1price[g] = profit
    if abs(profit) > 1e-3 or abs(p_gu) > 1e-3:
        print(f"  {g}: profit={profit:.2f} EUR")
        print(f"       {formula} = {profit:.2f}")

# Wind farms — 1-price: all deviations settled at λ_BM
for wi, w in enumerate(wind_names):
    dev     = dev_wind[w]
    da_prof = clean(da_profits.get(w, 0))

    if dev < 0:
        # eq (30): underproduced → penalised at BP
        bm_profit = clean(- lambda_BM * abs(dev))
        formula   = f"1-price → BP penalty = -{lambda_BM:.2f}*{abs(dev):.2f}"
    else:
        # eq (31): overproduced → rewarded at BP (this is the gaming incentive!)
        bm_profit = clean(lambda_BM * dev)
        formula   = f"1-price → BP reward = +{lambda_BM:.2f}*{dev:.2f}"

    total_profit_1price[w] = da_prof + bm_profit
    print(f"  {w}: DA={da_prof:.2f}  BM={bm_profit:.2f}  total={total_profit_1price[w]:.2f} EUR")
    print(f"       BM formula: {formula} = {bm_profit:.2f}")

# ── TWO-PRICE SETTLEMENT ──────────────────────────────────────────────────────
print(f"\n{'='*60}")
print(f"TWO-PRICE SETTLEMENT")
print(f"  Worse than scheduled (same dir)      → λ_BM = {lambda_BM:.2f} EUR/MWh")
print(f"  Better than scheduled (opposite dir) → λ_DA = {lambda_DA:.2f} EUR/MWh")
print(f"{'='*60}")

total_profit_2price = {}

# Conventional generators — identical to 1-price (providers always paid BP)
for g in conv_names:
    ps_g = g_DA[g]
    p_gu = r_up[g].X if g in r_up else 0.0
    c_g  = gen_costs[g]

    if g == 'G8':
        profit  = clean(lambda_DA * ps_g - lambda_BM * ps_g)
        formula = f"DA*PS - BP*PS = {lambda_DA:.2f}*{ps_g:.2f} - {lambda_BM:.2f}*{ps_g:.2f}"
    else:
        profit  = clean(lambda_DA * ps_g + lambda_BM * p_gu - c_g * (ps_g + p_gu))
        formula = (f"DA*PS + BP*p_gu - C_g*(PS+p_gu) = "
                  f"{lambda_DA:.2f}*{ps_g:.2f} + {lambda_BM:.2f}*{p_gu:.2f} "
                  f"- {c_g:.2f}*({ps_g:.2f}+{p_gu:.2f})")

    total_profit_2price[g] = profit
    if abs(profit) > 1e-3 or abs(p_gu) > 1e-3:
        print(f"  {g}: profit={profit:.2f} EUR")
        print(f"       {formula} = {profit:.2f}")

# Wind farms — 2-price: underproducers penalised at BP, overproducers only get DA
for wi, w in enumerate(wind_names):
    dev     = dev_wind[w]
    da_prof = clean(da_profits.get(w, 0))

    if dev < 0:
        # same direction as system short → penalised at BP (same as 1-price)
        bm_profit = clean(- lambda_BM * abs(dev))
        formula   = f"same dir → BP penalty = -{lambda_BM:.2f}*{abs(dev):.2f}"
    else:
        # opposite direction → settled at DA only (no BP bonus under 2-price)
        bm_profit = clean(lambda_DA * dev)
        formula   = f"opp dir → DA only = +{lambda_DA:.2f}*{dev:.2f}"

    total_profit_2price[w] = da_prof + bm_profit
    print(f"  {w}: DA={da_prof:.2f}  BM={bm_profit:.2f}  total={total_profit_2price[w]:.2f} EUR")
    print(f"       BM formula: {formula} = {bm_profit:.2f}")

# ── COMPARISON ────────────────────────────────────────────────────────────────
print(f"\n{'='*60}")
print(f"COMPARISON: 1-price vs 2-price total profit")
print(f"{'='*60}")
print(f"{'Agent':<10} {'1-price':>12} {'2-price':>12} {'diff':>10}")
print(f"{'-'*46}")
for g in list(conv_names) + list(wind_names):
    p1 = total_profit_1price.get(g, 0)
    p2 = total_profit_2price.get(g, 0)
    if abs(p1) > 1e-3 or abs(p2) > 1e-3:
        print(f"  {g:<8} {p1:>12.2f} {p2:>12.2f} {p2-p1:>10.2f}")

λ_DA  = 9.47 EUR/MWh
λ_BM  = 10.63 EUR/MWh
System imbalance: -421.67 MW (SHORT - upward needed)

Deviations from DA schedule:
  G8: -400.00 MW
  W1: -19.74 MW
  W2: -21.49 MW
  W3: -21.38 MW
  W4: +12.47 MW
  W5: +14.49 MW
  W6: +14.00 MW

ONE-PRICE SETTLEMENT (λ_BM = 10.63 EUR/MWh)
  G1: profit=-146.07 EUR
       DA*PS + BP*p_gu - C_g*(PS+p_gu) = 9.47*0.00 + 10.63*152.00 - 11.59*(0.00+152.00) = -146.07
  G2: profit=-22.27 EUR
       DA*PS + BP*p_gu - C_g*(PS+p_gu) = 9.47*0.00 + 10.63*23.18 - 11.59*(0.00+23.18) = -22.27
  G6: profit=49.60 EUR
       DA*PS + BP*p_gu - C_g*(PS+p_gu) = 9.47*155.00 + 10.63*0.00 - 9.15*(155.00+0.00) = 49.60
  G7: profit=49.60 EUR
       DA*PS + BP*p_gu - C_g*(PS+p_gu) = 9.47*155.00 + 10.63*0.00 - 9.15*(155.00+0.00) = 49.60
  G8: profit=-463.60 EUR
       DA*PS - BP*PS = 9.47*400.00 - 10.63*400.00 = -463.60
  G9: profit=1884.00 EUR
       DA*PS + BP*p_gu - C_g*(PS+p_gu) = 9.47*400.00 + 10.63*0.00 - 4.76*(400.00+0.00) = 1884.00
  G10: profit=2841.00 EUR
     

**PLOTS**

In [6]:
print(f"\nλ_DA = {lambda_DA:.2f} EUR/MWh  |  λ_BM = {lambda_BM:.2f} EUR/MWh")
print(f"System imbalance: {total_imbalance:+.2f} MW (SHORT — upward regulation needed)\n")

# --- Build table data ---
header = f"{'Generator':<12} {'DA Sched':>10} {'BM Act.':>10} {'Prod. Cost':>12} {'1-price':>12} {'2-price':>12} {'Diff':>10}"
sep    = '-' * len(header)

print(header)
print(sep)

for g in list(conv_names) + list(wind_names):
    p1 = total_profit_1price.get(g, 0)
    p2 = total_profit_2price.get(g, 0)

    # skip agents with zero profit in both schemes
    if abs(p1) < 1e-3 and abs(p2) < 1e-3:
        continue

    da_sched = g_DA.get(g, Wind_DA[wind_names.index(g)] if g in wind_names else 0)

    # BM activation
    if g in r_up:
        bm_act = f"+{r_up[g].X:.1f}"
    elif g == 'G8':
        bm_act = f"{-g_DA['G8']:.1f}"
    elif g in wind_names:
        wi  = wind_names.index(g)
        dev = Wind_actual[wi] - Wind_DA[wi]
        bm_act = f"{dev:+.1f}"
    else:
        bm_act = "0.0"

    cost = gen_costs.get(g, 0)
    diff = p2 - p1

    print(f"  {g:<10} {da_sched:>10.1f} {bm_act:>10} {cost:>12.2f} {p1:>12.2f} {p2:>12.2f} {diff:>10.2f}")

print(sep)

# Totals
total_1 = sum(total_profit_1price.get(g, 0) for g in list(conv_names) + list(wind_names))
total_2 = sum(total_profit_2price.get(g, 0) for g in list(conv_names) + list(wind_names))
print(f"  {'TOTAL':<10} {'':>10} {'':>10} {'':>12} {total_1:>12.2f} {total_2:>12.2f} {total_2-total_1:>10.2f}")
print(sep)

print(f"\nKey finding: W4/W5/W6 earn more under 1-price (settled at λ_BM={lambda_BM:.2f})")
print(f"             vs 2-price (settled at λ_DA={lambda_DA:.2f}) for their accidental overproduction.")
print(f"             This windfall incentivises strategic overproduction under 1-price.")


λ_DA = 9.47 EUR/MWh  |  λ_BM = 10.63 EUR/MWh
System imbalance: -421.67 MW (SHORT — upward regulation needed)

Generator      DA Sched    BM Act.   Prod. Cost      1-price      2-price       Diff
------------------------------------------------------------------------------------
  G1                0.0     +152.0        11.59      -146.07      -146.07       0.00
  G2                0.0      +23.2        11.59       -22.27       -22.27       0.00
  G6              155.0        0.0         9.15        49.60        49.60       0.00
  G7              155.0        0.0         9.15        49.60        49.60       0.00
  G8              400.0     -400.0         5.24      -463.60      -463.60       0.00
  G9              400.0        0.0         4.76      1884.00      1884.00       0.00
  G10             300.0        0.0         0.00      2841.00      2841.00       0.00
  G11             310.0        0.0         9.15        99.20        99.20       0.00
  G12             103.5     +246.5     